In [1]:
import os
os.chdir("../")

In [2]:
import numpy as np  # NPZ 읽기만 사용
import torch
from PIL import Image
from tqdm.auto import tqdm
from utils.fid import FIDInception  # 기존 클래스

NPZ  = '/dataset/dit/VIRTUAL_imagenet256_labeled.npz'  # arr_0 포함
OUT  = '/dataset/dit/valid_feats.pt'                   # 전체 feature 저장 파일
B    = 1024                                            # 배치 크기

# 1) 레퍼런스 로드 (메모리 절약: mmap_mode='r')
arr = np.load(NPZ, mmap_mode='r')['arr_0']  # (N,256,256,3) uint8 [0..255]
N = int(arr.shape[0])

# 2) Inception 준비 (GPU 사용 권장)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fid = FIDInception(dims=2048, normalize_input=True).to(device).eval()
for p in fid.parameters():
    p.requires_grad_(False)

# 3) Torch 텐서로 직접 기록 (CPU측 [N,2048] 버퍼 미리 할당)
feats = torch.empty((N, 2048), dtype=torch.float32, device='cpu')

# 4) 특징 추출 루프 (Torch만 사용해 feats에 기록)
with torch.inference_mode():
    for i in tqdm(range(0, N, B), total=(N + B - 1)//B, desc="Extracting Inception features"):
        j = min(i + B, N)
        # PIL 배치 생성 (fid는 PIL 리스트 입력)
        pil_batch = [Image.fromarray(arr[k], 'RGB') for k in range(i, j)]
        f = fid(pil_batch)              # [b,2048], device=device, float32
        feats[i:j].copy_(f.detach().cpu())  # Torch 텐서 슬라이스에 직접 기록 (numpy 사용 X)

# 5) Torch로 저장
torch.save({'features': feats}, OUT)
print(f"saved: {OUT}, shape={tuple(feats.shape)}, dtype={feats.dtype}, device={feats.device}")


/home/scpark/miniconda3/envs/dual/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting Inception features:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_245505/2469001333.py:29: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  pil_batch = [Image.fromarray(arr[k], 'RGB') for k in range(i, j)]
Extracting Inception features: 100%|██████████| 10/10 [00:19<00:00,  1.96s/it]

saved: /dataset/dit/valid_feats.pt, shape=(10000, 2048), dtype=torch.float32, device=cpu


In [5]:
!ls /dataset/dit/ -l

total 2201700
-rw-rw-r-- 1 scpark scpark   33572501 Aug 31 22:29 stats.pt
drwxrwxr-x 2 scpark scpark     262144 Aug 31 22:29 train1.5_10k_traj
drwxrwxr-x 2 scpark scpark      28672 Aug 31 22:29 train1.5_1k
drwxrwxr-x 2 scpark scpark      24576 Aug 31 22:25 train1.5_1k_traj
-rw-rw-r-- 1 scpark scpark   31312642 Aug 31 22:29 train1.5_1k.zip
drwxrwxr-x 2 scpark scpark       4096 Aug 31 22:24 valid1.5_100
-rw-rw-r-- 1 scpark scpark    3128920 Aug 31 22:29 valid1.5_100.zip
-rw-rw-r-- 1 scpark scpark   81921605 Sep  8 09:18 valid_feats.pt
-rw-rw-r-- 1 scpark scpark   33571316 Aug 31 22:29 valid_stats.npz
-rw-rw-r-- 1 scpark scpark   33572677 Aug 31 22:29 valid_stats.pt
-rw-rw-r-- 1 scpark scpark 2037122530 Aug 31 22:29 VIRTUAL_imagenet256_labeled.npz
